In [1]:
from dotenv import load_dotenv
import os
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from qMLPTabTransformer import run_experiment, infer_tabtransformer
from sklearn.metrics import precision_recall_curve, average_precision_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

%load_ext autoreload
%autoreload 2

In [ ]:
#Set up .env file with Kaggle API Token
load_dotenv()
os.environ["KAGGLE_API_TOKEN"] = os.getenv("KAGGLE_API_TOKEN")
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
file = os.path.join(path, 'creditcard.csv')
data = pd.read_csv(file)

print(f"# of Frauds = {np.sum(data['Class'])}")
print(np.sum(data['Class'])/len(data['Class'])*100, '%')

In [ ]:
#Split data into fraud and non-fraud to build high concentration of fraud in small training dataset
train_size = 2000
fraud_data = data[data['Class'] == 1].sample(frac=1, random_state=42)
non_fraud_data = data[data['Class'] == 0].sample(frac=1, random_state=42)
data_subset = pd.concat([fraud_data, non_fraud_data[:train_size - len(fraud_data)]]).sample(frac=1, random_state=42)

#Test model on small dataset
model, test_results, test_df = run_experiment(data_subset, lr=3e-3)
print(f'Test AUPRC: {test_results["auprc"]}')

Epoch 01 | train_loss=0.17321 | val_auprc=0.92328 | val_auroc=0.94945
Epoch 02 | train_loss=0.14918 | val_auprc=0.92984 | val_auroc=0.95553
Epoch 03 | train_loss=0.11057 | val_auprc=0.93488 | val_auroc=0.96175
Epoch 04 | train_loss=0.08236 | val_auprc=0.95737 | val_auroc=0.97567
Epoch 05 | train_loss=0.06208 | val_auprc=0.96586 | val_auroc=0.98216
Epoch 06 | train_loss=0.05179 | val_auprc=0.96811 | val_auroc=0.98378
Epoch 07 | train_loss=0.05543 | val_auprc=0.96271 | val_auroc=0.98013
Epoch 08 | train_loss=0.04773 | val_auprc=0.96009 | val_auroc=0.97702
Epoch 09 | train_loss=0.05057 | val_auprc=0.95847 | val_auroc=0.97567
Epoch 10 | train_loss=0.03860 | val_auprc=0.95754 | val_auroc=0.97581
Epoch 11 | train_loss=0.04513 | val_auprc=0.96181 | val_auroc=0.97986
Epoch 12 | train_loss=0.04580 | val_auprc=0.96196 | val_auroc=0.98040
Epoch 13 | train_loss=0.04692 | val_auprc=0.96233 | val_auroc=0.98000
Epoch 14 | train_loss=0.03809 | val_auprc=0.96144 | val_auroc=0.97919
Epoch 15 | train_los

In [ ]:
#Two level grid search for optimal learning rate 
test_auprcs = []
lrs = [1e-5, 1e-4, 1e-3, 1e-2]
best_auprc = 0
best_lr = 0
for lr in lrs:
    for seed in range(3):
        data_subset = pd.concat([fraud_data, non_fraud_data.sample(n=train_size-len(fraud_data), random_state=seed)]).sample(frac=1, random_state=seed)
        model, test_results, test_df = run_experiment(data_subset, lr=lr)
        test_auprc = test_results['auprc']
        test_auprcs.append(test_auprc)
        if test_auprc > best_auprc:
            best_auprc = test_auprc
            best_lr = lr
    print(np.mean(np.array(test_auprcs)), lr)

print()
test_auprcs = []
lrs = [best_lr*.3, best_lr*0.7, best_lr*3, best_lr*7]
for lr in lrs:
    for seed in range(3):
        data_subset = pd.concat([fraud_data, non_fraud_data.sample(n=train_size-len(fraud_data), random_state=seed)]).sample(frac=1, random_state=seed)
        model, test_results, test_df = run_experiment(data_subset, lr=lr)
        test_auprc = test_results['auprc']
        test_auprcs.append(test_auprc)
    print(np.mean(np.array(test_auprcs)), lr)

In [ ]:
#Tabulate total parameters and training parameters to see model size
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

Total params: 12,465
Trainable params: 12,465
